In [6]:
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    # Neo4j connection
    neo4j_uri: str
    neo4j_user: str
    neo4j_password: str

    # Paths
    pdf_path: str = "data/source/essentials-of-human-diseases-and-conditions_compress.pdf"
    processed_dir: str = "data/processed"

    class Config:
        env_file = "C:\\Users\\User\\Desktop\\agentic-graphrag-med\\data\\source\\.env"

settings = Settings()


C:\Users\User\AppData\Local\Temp\ipykernel_11412\3144254656.py:3: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class Settings(BaseSettings):


In [9]:
from neo4j import GraphDatabase


def get_driver():
    return GraphDatabase.driver(
        settings.neo4j_uri,
        auth=(settings.neo4j_user, settings.neo4j_password),
    )
get_driver()

In [3]:
from pathlib import Path
from typing import List

from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.document_loaders.text import TextLoader
from langchain_core.documents import Document  # type: ignore


def process_all_documents(data_directory: str) -> List[Document]:
    """
    Process all PDF and TXT files in a directory (recursively), like in your notebook.

    Returns a flat list of LangChain Document objects with metadata:
      - page (for PDFs, when loaded in page mode)
      - source (file path)
    """
    all_documents: List[Document] = []
    data_dir = Path(data_directory)

    # Find all PDF and TXT files recursively
    files = list(data_dir.rglob("*.pdf")) + list(data_dir.rglob("*.txt"))

    print(f"📂 Found {len(files)} files to process in {data_dir}")

    for file_path in files:
        try:
            if file_path.suffix.lower() == ".pdf":
                # Use mode="page" to keep page-level metadata, as recommended :contentReference[oaicite:4]{index=4}
                loader = PyPDFLoader(str(file_path), mode="page")
                docs = loader.load()
                print(f"  ✅ {file_path.name}: {len(docs)} pages loaded")
                all_documents.extend(docs)
            elif file_path.suffix.lower() == ".txt":
                loader = TextLoader(str(file_path), encoding="utf-8")
                docs = loader.load()
                print(f"  ✅ {file_path.name}: {len(docs)} text docs loaded")
                all_documents.extend(docs)
        except Exception as e:
            print(f"  ❌ Error loading {file_path.name}: {e}")

    print(f"📄 Total documents loaded: {len(all_documents)}")
    return all_documents
process_all_documents("C:\\Users\\User\\Desktop\\agentic-graphrag-med\\data\\source")

📂 Found 1 files to process in C:\Users\User\Desktop\agentic-graphrag-med\data\source
  ✅ essentials-of-human-diseases-and-conditions_compress.pdf: 1043 pages loaded
📄 Total documents loaded: 1043


[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Elsevier', 'creationdate': '2015-09-14T02:14:28-05:00', 'crossmarkdomains[1]': 'elsevier.com', 'crossmarkdomains[2]': 'sciencedirect.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'ebx_publisher': '/Elsevier Health Sciences', 'elsevierbookpdfspecifications': '1.32', 'moddate': '2020-06-27T10:01:24-05:00', 'robots': 'noindex', 'source': 'C:\\Users\\User\\Desktop\\agentic-graphrag-med\\data\\source\\essentials-of-human-diseases-and-conditions_compress.pdf', 'total_pages': 1043, 'page': 0, 'page_label': 'cover'}, page_content=''),
 Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Elsevier', 'creationdate': '2015-09-14T02:14:28-05:00', 'crossmarkdomains[1]': 'elsevier.com', 'crossmarkdomains[2]': 'sciencedirect.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'ebx_publisher': '/Elsevier Health Sciences', 'elsevierbookpdfspecificat

In [4]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import List

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document  # type: ignore

#from med_graphrag.config.settings import settings


def build_text_splitter(
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
) -> RecursiveCharacterTextSplitter:
    """
    Create a RecursiveCharacterTextSplitter with good defaults.

    Best-practice defaults for RAG are usually ~500–1000 chars chunk size
    and 50–200 overlap, depending on docs and model limits. :contentReference[oaicite:5]{index=5}
    """
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )


def generate_chunk_id(doc: Document, chunk_index: int) -> str:
    """
    Generate a stable unique ID for each chunk (idea inspired by similar
    patterns in many RAG blogs). :contentReference[oaicite:6]{index=6}
    """
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", 0)

    content_hash = hashlib.md5(doc.page_content.encode("utf-8")).hexdigest()[:8]
    return f"{Path(source).stem}_page{page}_chunk{chunk_index}_{content_hash}"


def split_documents(
    documents: List[Document],
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
) -> List[Document]:
    """
    Split documents into smaller chunks using RecursiveCharacterTextSplitter.
    Adds useful metadata for each chunk.
    """
    text_splitter = build_text_splitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    print("✂️  Splitting documents into chunks...")
    all_chunks: List[Document] = []

    for doc in documents:
        # split_documents expects a list of Document objects
        chunks = text_splitter.split_documents([doc])

        for i, chunk in enumerate(chunks):
            # enrich metadata
            chunk.metadata.setdefault("source", doc.metadata.get("source"))
            chunk.metadata.setdefault("page", doc.metadata.get("page"))

            chunk.metadata["chunk_index"] = i
            chunk.metadata["total_chunks_in_doc"] = len(chunks)
            chunk.metadata["chunk_id"] = generate_chunk_id(chunk, i)

            all_chunks.append(chunk)

    print(f"  → Created {len(all_chunks)} chunks")
    return all_chunks


def save_chunks_jsonl(chunks: List[Document], output_path: Path | None = None) -> Path:
    """
    Save chunks as JSONL with fields:
      - text
      - metadata (includes page, source, chunk_id, etc.)
    """
    if output_path is None:
        output_dir = Path(settings.processed_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        output_path = output_dir / "essentials_chunks.jsonl"

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as f:
        for ch in chunks:
            record = {
                "text": ch.page_content,
                "metadata": ch.metadata,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"💾 Chunks saved to: {output_path}")
    return output_path
